# Source Coverage Analysis — Myanmar ACLED

**Objective:** Quantify and visualise reporting bias in the ACLED Myanmar dataset, then apply a principled normalisation to reduce the effect of expanding media coverage over time.

## The problem in one sentence

The number of events recorded in ACLED grew sharply after the February 2021 coup — but part of that growth reflects *more journalists watching Myanmar*, not necessarily *more conflict*. If we use raw event counts as our dependent or independent variable in a model, we risk conflating conflict intensity with media attention.

## The signal we can use

Each ACLED event has a `source` field listing every outlet that reported on it (semicolon-separated). We can use the number of **unique source outlets** active in a given region-month as a proxy for *observation capacity* — how hard the media was looking — and divide event counts by that capacity to get a coverage-adjusted figure.

**Analogy:** In fisheries biology, *Catch Per Unit Effort* (CPUE) divides the number of fish caught by the fishing effort (hours, lines, etc.) to estimate fish abundance rather than fishing intensity. We do the same: events caught ÷ observation effort = coverage-adjusted conflict intensity.

---
**Note on what this normalisation does and does not fix:**
- ✓ Removes the time trend introduced by the growing number of outlets covering Myanmar post-2021
- ✓ Partially corrects for the fact that some regions attract more media attention than others
- ✗ Does **not** recover events that were never reported at all (systematic omission in low-access areas)
- ✗ Does **not** make low-coverage regions like rural Sagaing equivalent to well-covered Rakhine

Use normalised counts as a **robustness check** alongside raw counts, not as a definitive correction.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

CKPT_PATH = Path('checkpoints/myanmar_acled_grouped.parquet')

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

# Top admin1 regions we will focus on throughout the notebook
TOP_ADMIN1 = ['Sagaing', 'Magway', 'Mandalay', 'Rakhine', 'Kachin', 'Shan-North', 'Yangon', 'Tanintharyi']
ADMIN1_COLORS = px.colors.qualitative.Set2[:len(TOP_ADMIN1)]
COLOR_MAP = dict(zip(TOP_ADMIN1, ADMIN1_COLORS))

COUP_DATE = pd.Timestamp('2021-02-01')

## 2. Load data and parse the source field

We load the grouped checkpoint from `00_exploration_data.ipynb` and derive three coverage variables from the `source` column:

| Column | Definition |
|---|---|
| `n_sources` | Number of outlets that reported *this specific event* (corroboration count) |
| `source_list` | Parsed list of outlet names for this event |
| `month` | Event month as a Period (for groupby) |

These three columns are all we need for the coverage analysis.

In [2]:
df = pd.read_parquet(CKPT_PATH)

# Parse the semicolon-separated source field
df['n_sources']   = df['source'].str.count(';') + 1
df['source_list'] = df['source'].str.split(';').apply(lambda x: [s.strip() for s in x])
df['month']       = df['event_date'].dt.to_period('M')

print(f'Events: {len(df):,}')
print(f'Date range: {df["event_date"].min().date()} → {df["event_date"].max().date()}')
print(f'Unique outlets ever: {df["source_list"].explode().nunique()}')
print()
print('n_sources distribution:')
print(df['n_sources'].value_counts().sort_index().to_string())

Events: 105,463
Date range: 2010-01-01 → 2026-04-17
Unique outlets ever: 179

n_sources distribution:
n_sources
1     73662
2     18969
3      6638
4      2946
5      1434
6       749
7       455
8       275
9       170
10       96
11       33
12       21
13        5
14        6
15        3
18        1


## 3. National coverage trend over time

We aggregate to monthly resolution at the national level and compute three parallel series:

- **Raw event count**: the direct count of ACLED rows per month
- **Unique outlets active**: how many distinct source outlets reported at least one event that month — this is our *observation capacity* index
- **Mean n_sources per event**: average corroboration per event (how often each event was confirmed by multiple outlets)

If the post-2021 spike in events is purely a conflict increase, we expect `unique_outlets` to stay flat while `n_events` rises. If it also reflects a coverage expansion, both series should rise together. The correlation tells us how much of the event trend is a coverage artefact.

In [3]:
# Build national monthly summary
monthly_events = df.groupby('month').agg(
    n_events      = ('event_id_cnty', 'count'),
    total_sources = ('n_sources', 'sum'),
    mean_sources  = ('n_sources', 'mean'),
).reset_index()

# Unique outlets per month requires exploding the source_list
monthly_outlets = (
    df.groupby('month')['source_list']
      .apply(lambda x: pd.Series([s for sub in x for s in sub]).nunique())
      .reset_index(name='unique_outlets')
)

monthly = monthly_events.merge(monthly_outlets, on='month')
monthly['month_dt'] = monthly['month'].dt.to_timestamp()  # for plotting

r_events_outlets = monthly['n_events'].corr(monthly['unique_outlets'])
r_events_mean    = monthly['n_events'].corr(monthly['mean_sources'])
print(f'Pearson r(events, unique_outlets):  {r_events_outlets:.3f}')
print(f'Pearson r(events, mean_n_sources):  {r_events_mean:.3f}')
print()
print('A high r(events, unique_outlets) means more outlets → more events recorded,'
      ' i.e. coverage is inflating the count.')

Pearson r(events, unique_outlets):  0.829
Pearson r(events, mean_n_sources):  0.696

A high r(events, unique_outlets) means more outlets → more events recorded, i.e. coverage is inflating the count.


In [4]:
# Plot 1: events vs unique outlets over time (dual axis)
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(x=monthly['month_dt'], y=monthly['n_events'],
               name='Events / month', line=dict(color='steelblue', width=2)),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly['month_dt'], y=monthly['unique_outlets'],
               name='Unique outlets / month', line=dict(color='crimson', width=2, dash='dot')),
    secondary_y=True
)
fig.add_vline(x=str(COUP_DATE.date()), line_dash='dash', line_color='black', line_width=1)

fig.update_layout(
    title=f'Myanmar — Monthly events vs unique outlets (r = {r_events_outlets:.2f})',
    hovermode='x unified', width=1200, height=500,
    legend=dict(orientation='h', y=1.08)
)
fig.update_yaxes(title_text='Events per month', secondary_y=False)
fig.update_yaxes(title_text='Unique source outlets', secondary_y=True, showgrid=False)
fig.show()

In [5]:
# Plot 2: mean n_sources per event over time
# This tells us whether individual events are being better corroborated over time.
# A rising mean_sources suggests not just more events but each event is reported
# by more outlets — a sign that media attention per event is also growing.

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=monthly['month_dt'], y=monthly['mean_sources'],
    mode='lines', name='Mean n_sources per event',
    line=dict(color='darkorange', width=2)
))
fig2.add_vline(x=str(COUP_DATE.date()), line_dash='dash', line_color='black', line_width=1)
fig2.update_layout(
    title='Myanmar — Average corroboration per event over time',
    xaxis_title='Date', yaxis_title='Mean sources per event',
    width=1200, height=400, hovermode='x unified'
)
fig2.show()

pre  = monthly[monthly['month_dt'] < COUP_DATE]['mean_sources'].mean()
post = monthly[monthly['month_dt'] >= COUP_DATE]['mean_sources'].mean()
print(f'Mean sources per event — pre-coup:  {pre:.2f}')
print(f'Mean sources per event — post-coup: {post:.2f}')
print(f'Ratio: {post/pre:.2f}x — each event post-coup is corroborated by {post/pre:.2f}x as many outlets')

Mean sources per event — pre-coup:  1.14
Mean sources per event — post-coup: 1.63
Ratio: 1.42x — each event post-coup is corroborated by 1.42x as many outlets


## 4. Coverage by admin1 region

The national picture hides important regional variation. We now break coverage down by admin1 region to answer:

1. Which regions are systematically better covered (more outlets per event)?
2. Did the post-coup coverage expansion reach all regions equally?
3. Is there a structural gap between, say, Rakhine (where the Arakan Army has professional communications) and Sagaing (where PDF units are locally organised with minimal media access)?

We compute two metrics per admin1-month:
- **`unique_regional_outlets`**: distinct outlets that reported *any event from this admin1* in this month
- **`mean_n_sources`**: average per-event corroboration in this admin1-month

In [6]:
# Build admin1 × month coverage table
regional_outlets = (
    df.groupby(['admin1', 'month'])['source_list']
      .apply(lambda x: pd.Series([s for sub in x for s in sub]).nunique())
      .reset_index(name='unique_regional_outlets')
)

regional = (
    df.groupby(['admin1', 'month'])
      .agg(
          n_events     = ('event_id_cnty', 'count'),
          mean_sources = ('n_sources', 'mean'),
      )
      .reset_index()
      .merge(regional_outlets, on=['admin1', 'month'])
)
regional['month_dt'] = regional['month'].dt.to_timestamp()

# Restrict to top regions for clarity
reg_top = regional[regional['admin1'].isin(TOP_ADMIN1)].copy()

print('Mean unique regional outlets per month (averaged across all months):')
print(
    reg_top.groupby('admin1')['unique_regional_outlets']
           .mean().sort_values(ascending=False)
           .apply(lambda x: f'{x:.1f}')
)

Mean unique regional outlets per month (averaged across all months):
admin1
Shan-North     9.1
Magway         8.7
Sagaing        8.6
Tanintharyi    7.7
Kachin         7.5
Rakhine        7.1
Mandalay       7.1
Yangon         6.1
Name: unique_regional_outlets, dtype: str


In [7]:
# Plot 3: unique regional outlets over time per admin1
fig3 = go.Figure()
for admin1 in TOP_ADMIN1:
    sub = reg_top[reg_top['admin1'] == admin1]
    fig3.add_trace(go.Scatter(
        x=sub['month_dt'], y=sub['unique_regional_outlets'],
        name=admin1, mode='lines',
        line=dict(color=COLOR_MAP[admin1], width=1.8)
    ))
fig3.add_vline(x=str(COUP_DATE.date()), line_dash='dash', line_color='black', line_width=1)
fig3.update_layout(
    title='Unique source outlets per admin1 region per month',
    xaxis_title='Date', yaxis_title='Unique outlets',
    hovermode='x unified', width=1200, height=500,
    legend=dict(orientation='v', x=1.02, y=0.5)
)
fig3.show()

In [8]:
# Plot 4: heatmap — admin1 × year, coloured by mean n_sources per event
# This shows the structural gap between regions at a glance.
reg_top['year'] = reg_top['month'].dt.year
heatmap_data = (
    reg_top.groupby(['admin1', 'year'])['mean_sources']
           .mean().reset_index()
           .pivot(index='admin1', columns='year', values='mean_sources')
)

fig4 = go.Figure(go.Heatmap(
    z=heatmap_data.values.round(2),
    x=heatmap_data.columns.tolist(),
    y=heatmap_data.index.tolist(),
    colorscale='Blues',
    text=heatmap_data.values.round(2),
    texttemplate='%{text:.2f}',
    colorbar=dict(title='Mean n_sources')
))
fig4.update_layout(
    title='Mean sources per event — admin1 × year<br><sup>Higher = each event more corroborated = better covered region</sup>',
    xaxis_title='Year', yaxis_title='Admin1',
    width=1100, height=500
)
fig4.show()

## 5. The inflation problem: events correlate with coverage

Before normalising, we need to demonstrate that the problem is real — that raw event counts are partly driven by the number of outlets covering each region, not just by actual conflict activity.

We do this in two ways:
1. **Scatter plot**: for each admin1-month cell, plot events vs unique regional outlets. If these are correlated, coverage explains part of the event count.
2. **Time-series decomposition**: show that the slope of the events-over-time curve is steeper than we would expect if coverage were constant.

The intuition: if two regions have the same underlying conflict intensity but one has twice as many journalists, we would expect it to have more recorded events — even if nothing is actually different on the ground.

In [9]:
# Scatter: n_events vs unique_regional_outlets per admin1-month cell.
# Colour by admin1 to see whether the relationship is consistent across regions.
reg_top['month_str'] = reg_top['month'].astype(str)

fig5 = px.scatter(
    reg_top,
    x='unique_regional_outlets', y='n_events',
    color='admin1', color_discrete_map=COLOR_MAP,
    opacity=0.5,
    hover_data=['admin1', 'month_str'],
    labels={
        'unique_regional_outlets': 'Unique outlets covering this region this month',
        'n_events': 'Events recorded'
    },
    title='Events vs media coverage per admin1-month cell'
)
fig5.update_layout(width=1000, height=550)
fig5.show()

# Per-region correlation
print('Pearson r(events, unique_regional_outlets) per admin1:')
for admin1 in TOP_ADMIN1:
    sub = reg_top[reg_top['admin1'] == admin1]
    r = sub['n_events'].corr(sub['unique_regional_outlets'])
    print(f'  {admin1:<15s}  r = {r:.3f}')

Pearson r(events, unique_regional_outlets) per admin1:
  Sagaing          r = 0.851
  Magway           r = 0.941
  Mandalay         r = 0.912
  Rakhine          r = 0.827
  Kachin           r = 0.781
  Shan-North       r = 0.677
  Yangon           r = 0.757
  Tanintharyi      r = 0.902


## 6. Normalisation strategy

We want a normalised conflict intensity measure that answers: *"controlling for how hard the media was looking, how much conflict actually happened?"*

### Strategy: Coverage-Adjusted Event Rate (CAER)

We normalise each region-month's event count by the **number of unique source outlets active nationally that month**. The rationale for using the national (not regional) denominator:

- **Regional** unique outlets are endogenous: more conflict → more reporters sent there → higher denominator → over-corrects
- **National** unique outlets track the overall growth of Myanmar-focused media (new agencies, expanded coverage) while being less sensitive to region-specific events
- All regions in the same month get the same denominator, so cross-regional comparisons remain valid

$$\text{CAER}_{r,t} = \frac{\text{events}_{r,t}}{\text{national unique outlets}_t}$$

**Interpretation:** "Events in region *r* in month *t*, per outlet actively covering Myanmar that month." Holding the number of outlets constant, this removes the temporal inflation introduced by media expansion.

### What we are NOT doing

We are **not** using `events / n_sources_total` because `n_sources_total` (sum of per-event corroboration) conflates two things: (a) how many outlets exist, and (b) how significant each individual event was. A single spectacular battle with 10 outlets reporting it would inflate the denominator and artificially deflate other events in the same month.

We are **not** using regional unique outlets as the denominator because that creates circularity: regions with more fighting attract more reporters, so the denominator grows exactly where the numerator grows, producing a flat CAER for the most-conflicted areas regardless of true intensity.

In [10]:
# Build the national outlets index and merge into the regional table
national_outlets_idx = monthly[['month', 'unique_outlets']].rename(
    columns={'unique_outlets': 'national_unique_outlets'}
)

regional = regional.merge(national_outlets_idx, on='month')
regional['caer'] = regional['n_events'] / regional['national_unique_outlets']

# Sanity check
reg_top = regional[regional['admin1'].isin(TOP_ADMIN1)].copy()
reg_top['month_dt'] = reg_top['month'].dt.to_timestamp()

print('National unique outlets range: '
      f"{monthly['unique_outlets'].min()} (pre-2021) → {monthly['unique_outlets'].max()} (peak)")
print()
print('CAER interpretation: events per national outlet-month')
print()
print('Sample — Sagaing (raw events vs CAER):')
s = reg_top[reg_top['admin1'] == 'Sagaing'][[
    'month', 'n_events', 'national_unique_outlets', 'caer'
]].set_index('month').tail(6)
print(s.to_string())

National unique outlets range: 2 (pre-2021) → 50 (peak)

CAER interpretation: events per national outlet-month

Sample — Sagaing (raw events vs CAER):
         n_events  national_unique_outlets      caer
month                                               
2025-11       246                       32  7.687500
2025-12       231                       29  7.965517
2026-01       288                       33  8.727273
2026-02       174                       27  6.444444
2026-03       153                       29  5.275862
2026-04        86                       27  3.185185


## 7. Comparison: raw event counts vs CAER

We now visualise the effect of normalisation. The key questions:

1. Does the post-coup spike look smaller after normalisation? (It should, since part of the spike is coverage expansion.)
2. Does the ranking of regions change? (If Sagaing was undercounted, its CAER relative to other regions should increase.)
3. Does the correlation between events and coverage drop to near zero? (It should — that is exactly what we are removing.)

In [11]:
# Plot 6: national total — raw events vs CAER over time
national_caer = (
    regional.groupby('month')
            .agg(n_events=('n_events', 'sum'), caer=('caer', 'sum'))
            .reset_index()
)
national_caer['month_dt'] = national_caer['month'].dt.to_timestamp()

fig6 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['Raw event count (national total)',
                                     'CAER — events per national outlet-month'],
                     vertical_spacing=0.10)

fig6.add_trace(
    go.Scatter(x=national_caer['month_dt'], y=national_caer['n_events'],
               name='Raw events', fill='tozeroy',
               line=dict(color='steelblue', width=1.5)),
    row=1, col=1
)
fig6.add_trace(
    go.Scatter(x=national_caer['month_dt'], y=national_caer['caer'],
               name='CAER', fill='tozeroy',
               line=dict(color='darkorange', width=1.5)),
    row=2, col=1
)
for row in [1, 2]:
    fig6.add_vline(x=str(COUP_DATE.date()), line_dash='dash', line_color='black',
                   line_width=1, row=row, col=1)

fig6.update_layout(height=600, width=1200,
                   title='National conflict trend: raw vs coverage-adjusted',
                   hovermode='x unified', showlegend=False)
fig6.show()

# Quantify the difference in the post-coup spike
pre  = national_caer[national_caer['month_dt'] < COUP_DATE]
post = national_caer[national_caer['month_dt'] >= COUP_DATE]
raw_ratio  = post['n_events'].mean() / pre['n_events'].mean()
caer_ratio = post['caer'].mean()    / pre['caer'].mean()
print(f'Post/pre-coup ratio — raw events: {raw_ratio:.2f}x')
print(f'Post/pre-coup ratio — CAER:       {caer_ratio:.2f}x')
print(f'Coverage expansion accounts for ~{(1 - caer_ratio/raw_ratio)*100:.0f}% of the apparent spike')

Post/pre-coup ratio — raw events: 17.56x
Post/pre-coup ratio — CAER:       8.97x
Coverage expansion accounts for ~49% of the apparent spike


In [12]:
# Plot 7: per-region CAER over time (top admin1 only)
fig7 = make_subplots(
    rows=4, cols=2,
    subplot_titles=TOP_ADMIN1,
    shared_yaxes=False, shared_xaxes=False,
    vertical_spacing=0.10, horizontal_spacing=0.08
)

for i, admin1 in enumerate(TOP_ADMIN1):
    row, col = (i // 2) + 1, (i % 2) + 1
    sub = reg_top[reg_top['admin1'] == admin1]

    fig7.add_trace(
        go.Scatter(x=sub['month_dt'], y=sub['n_events'],
                   name='Raw', line=dict(color='steelblue', width=1.2),
                   showlegend=(i == 0)),
        row=row, col=col
    )
    # Scale CAER to same axis as raw events for visual comparison
    scale = sub['n_events'].max() / sub['caer'].max() if sub['caer'].max() > 0 else 1
    fig7.add_trace(
        go.Scatter(x=sub['month_dt'], y=sub['caer'] * scale,
                   name='CAER (scaled)', line=dict(color='darkorange', width=1.2, dash='dot'),
                   showlegend=(i == 0)),
        row=row, col=col
    )
    fig7.add_vline(x=str(COUP_DATE.date()), line_dash='dash', line_color='black',
                   line_width=0.8, row=row, col=col)

fig7.update_layout(
    height=1000, width=1200,
    title='Raw events (blue) vs CAER scaled to same axis (orange dashed) — by admin1<br>'
          '<sup>When orange is below blue: coverage inflation is pulling the raw count up. '
          'When they track closely: conflict explains the trend.</sup>',
    hovermode='x unified'
)
fig7.show()

In [13]:
# Plot 8: scatter — does CAER remove the coverage correlation?
# Compare: (raw events vs unique_regional_outlets) vs (CAER vs unique_regional_outlets)
fig8 = make_subplots(rows=1, cols=2,
                     subplot_titles=['Raw events vs regional outlets',
                                     'CAER vs regional outlets'])

for col_idx, (y_col, y_label) in enumerate([('n_events','Raw events'), ('caer','CAER')], start=1):
    for admin1 in TOP_ADMIN1:
        sub = reg_top[reg_top['admin1'] == admin1]
        fig8.add_trace(
            go.Scatter(
                x=sub['unique_regional_outlets'], y=sub[y_col],
                mode='markers', name=admin1,
                marker=dict(color=COLOR_MAP[admin1], opacity=0.5, size=5),
                showlegend=(col_idx == 1)
            ),
            row=1, col=col_idx
        )

fig8.update_xaxes(title_text='Unique regional outlets')
fig8.update_yaxes(title_text='Value', col=1)
fig8.update_layout(height=480, width=1200,
                   title='Does normalisation remove the coverage correlation?')
fig8.show()

# Print correlations
print('Correlation (events, unique_regional_outlets) — raw vs CAER:')
for admin1 in TOP_ADMIN1:
    sub = reg_top[reg_top['admin1'] == admin1]
    r_raw  = sub['n_events'].corr(sub['unique_regional_outlets'])
    r_caer = sub['caer'].corr(sub['unique_regional_outlets'])
    print(f'  {admin1:<15s}  raw: {r_raw:+.3f}  →  CAER: {r_caer:+.3f}')

Correlation (events, unique_regional_outlets) — raw vs CAER:
  Sagaing          raw: +0.851  →  CAER: +0.821
  Magway           raw: +0.941  →  CAER: +0.912
  Mandalay         raw: +0.912  →  CAER: +0.899
  Rakhine          raw: +0.827  →  CAER: +0.767
  Kachin           raw: +0.781  →  CAER: +0.609
  Shan-North       raw: +0.677  →  CAER: +0.592
  Yangon           raw: +0.757  →  CAER: +0.754
  Tanintharyi      raw: +0.902  →  CAER: +0.879


## 8. Save the normalised series

We save two artefacts:

1. **`source_coverage_monthly.csv`** — national monthly table with raw events, unique outlets, mean_sources, and CAER
2. **`source_coverage_regional.csv`** — admin1 × month table with raw events, coverage variables, and CAER

These can be joined to any downstream model on the `month` and `admin1` keys.

In [14]:
OUT_DIR = Path('coverage_outputs')
OUT_DIR.mkdir(exist_ok=True)

# National monthly
monthly_out = monthly.drop(columns=['month_dt']).copy()
monthly_out['month'] = monthly_out['month'].astype(str)
monthly_out['caer_national'] = monthly_out['n_events'] / monthly_out['unique_outlets']
monthly_out.to_csv(OUT_DIR / 'source_coverage_monthly.csv', index=False)
print(f'Saved: {OUT_DIR}/source_coverage_monthly.csv  ({len(monthly_out)} rows)')

# Regional admin1 × month
regional_out = regional[[
    'admin1', 'month', 'n_events', 'mean_sources',
    'unique_regional_outlets', 'national_unique_outlets', 'caer'
]].copy()
regional_out['month'] = regional_out['month'].astype(str)
regional_out.to_csv(OUT_DIR / 'source_coverage_regional.csv', index=False)
print(f'Saved: {OUT_DIR}/source_coverage_regional.csv  ({len(regional_out)} rows)')

print()
print('Column reference:')
print('  month                    — YYYY-MM period')
print('  n_events                 — raw event count')
print('  unique_regional_outlets  — distinct outlets covering this admin1 this month')
print('  national_unique_outlets  — distinct outlets covering Myanmar this month (denominator for CAER)')
print('  caer                     — Coverage-Adjusted Event Rate = n_events / national_unique_outlets')
print('  mean_sources             — mean per-event corroboration (different from CAER denominator)')

Saved: coverage_outputs/source_coverage_monthly.csv  (196 rows)
Saved: coverage_outputs/source_coverage_regional.csv  (2543 rows)

Column reference:
  month                    — YYYY-MM period
  n_events                 — raw event count
  unique_regional_outlets  — distinct outlets covering this admin1 this month
  national_unique_outlets  — distinct outlets covering Myanmar this month (denominator for CAER)
  caer                     — Coverage-Adjusted Event Rate = n_events / national_unique_outlets
  mean_sources             — mean per-event corroboration (different from CAER denominator)


## 9. Caveats and limitations

### What CAER corrects
- The **temporal inflation** caused by the growth of Myanmar-focused media after the 2021 coup. The post-coup spike in events is partly real conflict and partly more journalists. CAER separates these.
- **Cross-period comparisons**: a CAER value of 10 in 2019 is more comparable to a CAER of 10 in 2023 than the corresponding raw counts would be.

### What CAER does not correct
- **Systematic omission bias**: events that were never reported at all (no source, therefore never entered the database) cannot be recovered. Rural Sagaing PDFs operating without any media presence are structurally undercounted — CAER does not fix this.
- **Source quality variation**: a single BBC report is not equivalent to a single local activist Twitter post. CAER treats all outlets as equal observation units.
- **Regional coverage heterogeneity within a month**: in months where the Arakan Army's PR unit is active, Rakhine gets more coverage per event. The national denominator partially controls for this, but not fully.

### Recommendation for modelling
Run your main models on both raw event counts and CAER. If conclusions are qualitatively the same, report the raw results (easier to interpret) and note robustness to coverage normalisation. If conclusions differ, investigate whether the difference is driven by the post-coup period or by specific high-coverage regions (Rakhine, Shan-North).